# Pipeline — Pré-tratamento e features patrimoniais por candidato

Este notebook consolida a primeira fase do projeto: transformar a base bruta de bens declarados dos candidatos em uma base de **features patrimoniais por candidato**.

A unidade de entrada é um dataset de bens declarados pelos candidatos em uma eleição brasileira, esses podem ser encontrados no site do TSE(Tribunal Superior Eleitoral). Cada linha representa um bem declarado. A saída principal é o arquivo `features_patrimoniais_por_candidato.csv`, no qual cada linha representa um candidato com variáveis agregadas de patrimônio.

O pipeline foi organizado para ser **repetível e parametrizável**, mas com menos opções técnicas expostas ao usuário. A estrutura das colunas finais é fixa, a normalização numérica sempre é aplicada e a execução não compara o resultado com um arquivo de referência.


## 1. Parâmetros da geração

Altere esta célula antes de cada execução. Os parâmetros mais importantes para iteração são `NOME_RODADA`, `CAMINHO_BENS_UNIFICADO`, `AJUSTES_TIPO_MACRO`, `REGRAS_MANUAIS_TEXTO` e a seleção de `ARQUIVOS_SAIDA`.

Por padrão, todos os arquivos finais são gerados. Para não gerar algum deles, troque o respectivo valor em `ARQUIVOS_SAIDA` para `False`.


In [1]:
from pathlib import Path
import csv
import json
import re
import shutil
import unicodedata
from datetime import datetime

import numpy as np
import pandas as pd

# ============================================================
# Parâmetros principais da rodada
# ============================================================

PARAMS = {
    # Identificação da rodada. Troque este valor quando quiser uma nova execução sem sobrescrever saídas.
    "NOME_RODADA": "features_patrimoniais_por_candidato",

    # Caminho de entrada.
    "CAMINHO_BENS_UNIFICADO": "bem_candidato_2022_BRASIL.csv",

    # Saídas.
    "DIR_SAIDA_BASE": ".",

    # Leitura/escrita.
    # Use "auto" para detectar encoding/separador quando necessário.
    "SEP_ENTRADA": ";",
    "ENCODING_ENTRADA": "latin1",
    "DECIMAL_ENTRADA": ",",
    "SEP_SAIDA": ";",
    "ENCODING_SAIDA": "utf-8-sig",

    # Tratamento de texto.
    "CORRIGIR_MOJIBAKE": True,

    # Tratamento.
    "REMOVER_DUPLICADOS": True,
    "REMOVER_CANDIDATOS_PATRIMONIO_ZERO_FEATURES": True,
    "VALOR_PREENCHER_MACRO_NA": "outros",

    # Macros fixas esperadas no dataset final.
    # A ordem abaixo define sempre a ordem das colunas de saída.
    "MACROS_FEATURES_PADRAO": [
        "ativos_financeiros",
        "bens_luxo_colecao",
        "creditos_direitos",
        "dinheiro_especie",
        "direitos_intangiveis",
        "imoveis",
        "outros",
        "outros_atividade_profissional",
        "participacoes_societarias",
        "rural_agropecuario",
        "veiculos",
    ],

    # Metadados finais da base de features.
    "METADADOS_FEATURES": ["SG_UF"],

    # Auditoria.
    "N_AMOSTRAS_AUDITORIA": 50,
    "VALOR_MINIMO_AUDITORIA": 0,

    # Exportação.
    # Todos vêm ligados por padrão. Troque para False para não gerar algum arquivo/grupo.
    "ARQUIVOS_SAIDA": {
        "bens_tratados": True,
        "bens_limpo_com_macros": True,
        "features_candidatos": True,
        "features_final_raiz": True,
        "checklist": True,
        "auditorias": True,
        "config": True,
        "zip_rodada": False,
    },
}

# ============================================================
# Ajustes manuais por tipo oficial do TSE
# ============================================================
# Use quando um valor específico de DS_TIPO_BEM_CANDIDATO precisar ser forçado
# para uma macro categoria.

AJUSTES_TIPO_MACRO = {}

# ============================================================
# Regras manuais por texto
# ============================================================
# Cada regra é aplicada sobre texto normalizado formado por:
# DS_TIPO_BEM_CANDIDATO + DS_BEM_CANDIDATO.
#
# Campos:
# - nome: identificador da regra
# - padrao: regex em texto normalizado
# - macro: macro_categoria_refinada desejada
# - submacro_col: coluna de submacro a preencher opcionalmente
# - submacro_valor: valor opcional da submacro

REGRAS_MANUAIS_TEXTO = []

# ============================================================
# Checklist de aceite sugerido
# ============================================================
# Estes limites não bloqueiam a execução. Servem para leitura da auditoria.

CHECKLIST_ACEITE = {
    "max_tipos_sem_macro": 0,
    "max_perc_macro_outros": 10.0,
    "max_perc_valor_outros": 10.0,
    "max_perc_financeiro_generico": 20.0,
    "max_perc_imovel_generico": 20.0,
    "max_perc_veiculo_generico": 20.0,
}


## 2. Mapeamento base de tipos de bens para macro categorias

Esta é a classificação inicial por tipo oficial de bem. As funções seguintes refinam casos genéricos ou ambíguos a partir da descrição textual.

In [2]:
MAPA_TIPO_MACRO_BASE = {
    # Imóveis
    "Apartamento": "imoveis",
    "Casa": "imoveis",
    "Terreno": "imoveis",
    "Sala ou conjunto": "imoveis",
    "Galpão": "imoveis",
    "Loja": "imoveis",
    "Prédio comercial": "imoveis",
    "Prédio residencial": "imoveis",
    "Construção": "imoveis",
    "Benfeitorias": "imoveis",
    "Outros bens imóveis": "imoveis",

    # Rural / agropecuário
    "Terra nua": "rural_agropecuario",

    # Veículos
    "Veículo automotor terrestre: caminhão, automóvel, moto, etc.": "veiculos",
    "Embarcação": "veiculos",
    "Aeronave": "veiculos",

    # Ativos financeiros
    "Aplicação de renda fixa (CDB, RDB e outros)": "ativos_financeiros",
    "Caderneta de poupança": "ativos_financeiros",
    "Depósito bancário em conta corrente no País": "ativos_financeiros",
    "Depósito bancário em conta corrente no exterior": "ativos_financeiros",
    "VGBL - Vida Gerador de Benefício Livre": "ativos_financeiros",
    "Ações (inclusive as provenientes de linha telefônica)": "ativos_financeiros",
    "Fundo de Investimento Imobiliário": "ativos_financeiros",
    "Outros fundos": "ativos_financeiros",
    "Fundo de Curto Prazo": "ativos_financeiros",
    "Fundo de Longo Prazo e Fundo de Investimentos em Direitos Creditórios (FIDC)": "ativos_financeiros",
    "Fundos: Ações, Mútuos de Privatização, Invest. Empresas Emergentes, Invest.Participação e Invest. Índice Mercado": "ativos_financeiros",
    "Ouro, ativo financeiro": "ativos_financeiros",
    "Mercado futuros, de opções e a termo": "ativos_financeiros",
    "Plano PAIT e caderneta de pecúlio": "ativos_financeiros",
    "Outras aplicações e Investimentos": "ativos_financeiros",
    "Outros depósitos à vista e numerário": "ativos_financeiros",

    # Participações societárias
    "Quotas ou quinhões de capital": "participacoes_societarias",
    "Outras participações societárias": "participacoes_societarias",

    # Dinheiro
    "Dinheiro em espécie - moeda nacional": "dinheiro_especie",
    "Dinheiro em espécie - moeda estrangeira": "dinheiro_especie",

    # Créditos e direitos
    "Crédito decorrente de alienação": "creditos_direitos",
    "Crédito decorrente de empréstimo": "creditos_direitos",
    "Outros créditos e poupança vinculados": "creditos_direitos",
    "Consórcio não contemplado": "creditos_direitos",
    "Poupança para construção ou aquisição de bem imóvel": "creditos_direitos",
    "Leasing": "creditos_direitos",

    # Bens de luxo / coleção
    "Jóia, quadro, objeto de arte, de coleção, antiguidade, etc.": "bens_luxo_colecao",

    # Direitos intangíveis
    "Direito de autor, de inventor e patente": "direitos_intangiveis",
    "Direito de lavra e assemelhado": "direitos_intangiveis",
    "Licença e concessões especiais": "direitos_intangiveis",

    # Outros
    "Linha telefônica": "outros",
    "Título de clube e assemelhado": "outros",
    "Bem relacionado com o exercício da atividade autônoma": "outros",
    "Outros bens móveis": "outros",
    "OUTROS BENS E DIREITOS": "outros",
}

MAPA_TIPO_MACRO = {**MAPA_TIPO_MACRO_BASE, **AJUSTES_TIPO_MACRO}

## 3. Leitura robusta, correção de encoding e utilitários

Esta seção corrige os problemas de formatação textual observados (`Ã£`, `Ã©`, `ï»¿`) e detecta automaticamente separador/encoding.

In [3]:
def parece_mojibake(texto):
    """Identifica padrões comuns de texto lido com encoding errado."""
    if not isinstance(texto, str):
        return False
    marcadores = ("Ã", "Â", "â€", "â€“", "â€œ", "â€�", "ï»¿", "�")
    return any(m in texto for m in marcadores)


def remover_bom_colunas(df):
    """Remove BOM e espaços residuais dos nomes das colunas."""
    df = df.copy()
    df.columns = [str(c).replace("\ufeff", "").replace("ï»¿", "").strip() for c in df.columns]
    return df


def corrigir_mojibake_texto(x):
    """
    Corrige mojibake em textos como 'SÃ£o', 'AplicaÃ§Ã£o' e 'ï»¿'.
    A correção só é aplicada quando há marcadores claros de encoding errado.
    """
    if pd.isna(x):
        return x
    if not isinstance(x, str):
        return x

    s = x.replace("\ufeff", "").replace("ï»¿", "")

    for _ in range(3):
        if not parece_mojibake(s):
            break
        try:
            novo = s.encode("latin1", errors="strict").decode("utf-8", errors="strict")
        except Exception:
            break
        if novo == s:
            break
        s = novo.replace("\ufeff", "").replace("ï»¿", "")

    return s


def corrigir_mojibake_dataframe(df, params):
    """Aplica correção de mojibake em colunas textuais e nomes de colunas."""
    df = remover_bom_colunas(df)
    if not params.get("CORRIGIR_MOJIBAKE", True):
        return df

    df = df.copy()
    df.columns = [corrigir_mojibake_texto(c) for c in df.columns]

    colunas_texto = df.select_dtypes(include=["object", "string"]).columns
    for col in colunas_texto:
        df[col] = df[col].map(corrigir_mojibake_texto)

    return df


def detectar_sep(caminho, encoding):
    """Detecta separador entre ';' e ',' usando a primeira linha decodificada."""
    with open(caminho, "rb") as f:
        amostra = f.read(4096)
    texto = amostra.decode(encoding, errors="replace")
    primeira_linha = texto.splitlines()[0] if texto.splitlines() else ""
    if primeira_linha.count(";") >= primeira_linha.count(","):
        return ";"
    return ","


def score_mojibake_df(df):
    """Pontua a presença de mojibake em uma amostra de dataframe."""
    score = 0
    for col in df.columns:
        score += 5 if parece_mojibake(str(col)) else 0
    colunas_texto = df.select_dtypes(include=["object", "string"]).columns[:6]
    for col in colunas_texto:
        amostra = df[col].dropna().astype(str).head(200)
        score += sum(1 for x in amostra if parece_mojibake(x))
    return score


def ler_csv_robusto(caminho, params):
    """
    Lê CSV tentando evitar dois problemas comuns do projeto:
    1. Separador diferente entre bases intermediárias e arquivos brutos.
    2. Encoding incorreto, que gera textos como 'SÃ£o' ou 'AplicaÃ§Ã£o'.
    """
    caminho = Path(caminho)
    enc_param = params.get("ENCODING_ENTRADA", "auto")
    sep_param = params.get("SEP_ENTRADA", "auto")
    decimal_param = params.get("DECIMAL_ENTRADA", None)
    read_kwargs = {} if decimal_param is None else {"decimal": decimal_param}
    if str(caminho).endswith("bem_candidato_2022_BRASIL.csv"):
        read_kwargs["dtype"] = {"SQ_CANDIDATO": "Int64"}

    encodings = ["utf-8-sig", "utf-8", "cp1252", "latin1"] if enc_param == "auto" else [enc_param]
    candidatos = []
    erros = []

    for enc in encodings:
        try:
            sep = detectar_sep(caminho, enc) if sep_param == "auto" else sep_param
            df_amostra = pd.read_csv(caminho, sep=sep, encoding=enc, nrows=300, low_memory=False, **read_kwargs)
            score = score_mojibake_df(df_amostra)
            candidatos.append((score, enc, sep))
        except Exception as e:
            erros.append((enc, str(e)))

    if not candidatos:
        raise ValueError(f"Não foi possível ler {caminho}. Erros: {erros}")

    candidatos = sorted(candidatos, key=lambda x: x[0])
    _, encoding_escolhido, sep_escolhido = candidatos[0]

    df = pd.read_csv(caminho, sep=sep_escolhido, encoding=encoding_escolhido, low_memory=False, **read_kwargs)
    df = corrigir_mojibake_dataframe(df, params)
    return df, {"encoding": encoding_escolhido, "sep": sep_escolhido}


def normalizar_texto(x):
    """Normaliza texto para uso em regras baseadas em regex."""
    if pd.isna(x):
        return ""

    x = corrigir_mojibake_texto(x)
    x = str(x).lower()
    x = unicodedata.normalize("NFKD", x)
    x = "".join(c for c in x if not unicodedata.combining(c))
    x = re.sub(r"[^a-z0-9\s]", " ", x)
    x = re.sub(r"\s+", " ", x).strip()
    return x


def normalizar_valor_monetario(serie, params=None):
    """Converte valores monetários do TSE para numérico em todas as execuções."""
    s = serie.astype(str).str.strip()
    s = s.str.replace("\ufeff", "", regex=False).str.replace("ï»¿", "", regex=False)
    s = s.str.replace(".", "", regex=False).str.replace(",", ".", regex=False)
    s = s.str.replace(r"[^0-9\.\-]", "", regex=True)
    return pd.to_numeric(s, errors="coerce").fillna(0)


def localizar_coluna(df, candidatos):
    """Retorna a primeira coluna existente dentro de uma lista de nomes candidatos."""
    for col in candidatos:
        if col in df.columns:
            return col
    return None


def resumo_categoria(df, categoria_col="macro_categoria_refinada", valor_col="VR_BEM_CANDIDATO_NUM"):
    """Gera resumo por categoria, com contagem e estatísticas de valor."""
    if df.empty or categoria_col not in df.columns:
        return pd.DataFrame()

    out = (
        df.groupby(categoria_col)[valor_col]
        .agg(qtd="count", valor_total="sum", valor_mediano="median", valor_medio="mean")
        .sort_values("valor_total", ascending=False)
    )

    total_valor = out["valor_total"].sum()
    total_qtd = out["qtd"].sum()

    out["perc_valor_total"] = np.where(total_valor > 0, out["valor_total"] / total_valor * 100, 0).round(2)
    out["perc_qtd"] = np.where(total_qtd > 0, out["qtd"] / total_qtd * 100, 0).round(2)

    return out


def ler_base_bens(params):
    """Lê sempre a base unificada informada em CAMINHO_BENS_UNIFICADO."""
    caminho_unificado = Path(params["CAMINHO_BENS_UNIFICADO"])

    if not caminho_unificado.exists():
        raise FileNotFoundError(
            f"Não encontrei o arquivo unificado: {caminho_unificado}. "
            "Ajuste CAMINHO_BENS_UNIFICADO para apontar para o CSV de entrada."
        )

    df, info = ler_csv_robusto(caminho_unificado, params)
    origem = f"arquivo_unificado:{caminho_unificado} | encoding={info['encoding']} | sep={info['sep']}"
    return df, origem


def preparar_base_inicial(df, params):
    """Padroniza colunas essenciais e cria campos auxiliares."""
    df = corrigir_mojibake_dataframe(df, params)
    df = df.copy()

    col_valor = localizar_coluna(df, ["VR_BEM_CANDIDATO", "VR_BEM_CANDIDATO_NUM"])
    if col_valor is None:
        raise KeyError("A base precisa conter VR_BEM_CANDIDATO ou VR_BEM_CANDIDATO_NUM.")

    if "VR_BEM_CANDIDATO" not in df.columns:
        df["VR_BEM_CANDIDATO"] = df[col_valor]

    df["VR_BEM_CANDIDATO_NUM"] = normalizar_valor_monetario(df[col_valor], params)

    for col in ["DS_TIPO_BEM_CANDIDATO", "DS_BEM_CANDIDATO"]:
        if col not in df.columns:
            df[col] = ""

    df["tipo_norm"] = df["DS_TIPO_BEM_CANDIDATO"].apply(normalizar_texto)
    df["desc_norm"] = df["DS_BEM_CANDIDATO"].apply(normalizar_texto)
    df["texto_norm"] = (df["tipo_norm"].astype(str) + " " + df["desc_norm"].astype(str)).str.strip()

    df["macro_categoria"] = df["DS_TIPO_BEM_CANDIDATO"].map(MAPA_TIPO_MACRO)
    df["macro_categoria"] = df["macro_categoria"].fillna(params["VALOR_PREENCHER_MACRO_NA"])
    df["macro_categoria_refinada"] = df["macro_categoria"]

    total = len(df)
    dist = df["macro_categoria"].value_counts(normalize=True) * 100
    df["perc_registros"] = df["macro_categoria"].map(dist).round(2)

    if params["REMOVER_DUPLICADOS"]:
        subset_dup = [
            col for col in [
                "SQ_CANDIDATO",
                "NR_ORDEM_BEM_CANDIDATO",
                "DS_TIPO_BEM_CANDIDATO",
                "DS_BEM_CANDIDATO",
                "VR_BEM_CANDIDATO_NUM"
            ]
            if col in df.columns
        ]
        if subset_dup:
            df = df.drop_duplicates(subset=subset_dup).copy()
        else:
            df = df.drop_duplicates().copy()

    return df


## 4. Regras finais de refinamento

As funções abaixo consolidam as regras finais dos dois notebooks originais para imóveis, ativos financeiros, outros bens e veículos.

In [4]:
def classificar_submacro_imoveis(row):
    texto = f"{row['tipo_norm']} {row['desc_norm']}"

    padrao_participacao = (
        r"\b("
        r"participacao|participacoes|part|sociedade|socied|"
        r"capital social|quotas|quinhao|quinhoes|empreend|"
        r"empreendimento|empresa|ltda|limitada|cnpj"
        r")\b"
    )
    padrao_rural = (
        r"\b("
        r"rural|fazenda|faz|sitio|sitios|chacara|chacaras|"
        r"gleba|terra nua|propriedade rural|zona rural|area rural|"
        r"imovel rural|terras rurais|estancia|rancho|granja|"
        r"haras|aras|hectare|hectares"
        r")\b"
        r"|(\bha\b|\bhas\b)|ha de terras|has de terras"
    )
    padrao_comercial = r"\b(comercial|sala|salas|loja|lojas|galpao|galpoes|barracao|predio comercial|ponto comercial|shopping|empresarial|office|industrial|comercio|padaria|pousada|supermercado|hospital|hotel|posto|clinica|restaurante|escritorio)\b"
    padrao_construcao = r"\b(construcao|construcoes|benfeitoria|benfeitorias|obra|obras|andamento|reforma|edificacao|edificacoes|planta)\b"
    padrao_residencial = r"\b(casa|casas|apartamento|apartamentos|apto|apt|residencial|residencia|condominio|edificio|sobrado|alvenaria|moradia)\b"
    padrao_terreno = r"\b(terreno|terrenos|lote|lotes|quadra|loteamento|fracao|parte ideal)\b"
    padrao_urbano_generico = r"\b(urbano|imovel urbano|cidade|municipio|rua|avenida|bairro|distrito federal|sao paulo|porto alegre|miami|balneario)\b"

    if re.search(padrao_participacao, texto):
        return "participacoes_societarias"
    if re.search(padrao_rural, texto):
        return "imovel_rural"
    if re.search(padrao_comercial, texto):
        return "imovel_comercial_industrial"
    if re.search(padrao_construcao, texto):
        return "construcao_benfeitoria"
    if re.search(padrao_residencial, texto):
        return "imovel_residencial"
    if re.search(padrao_terreno, texto):
        return "terreno_lote"
    if re.search(padrao_urbano_generico, texto):
        return "imovel_urbano_generico"
    return "imovel_generico"


def classificar_submacro_financeira(row):
    tipo = row["tipo_norm"]
    desc = row["desc_norm"]
    texto = f"{tipo} {desc}"

    padrao_fundo_forte = r"\b(fundo|fundos|fii|fidc|fip|fic|fundo de investimento|fundos de investimento|fundo imobiliario|investimento imobiliario|multimercado|firf|fima|fim|cotas de fundos|cotas do fundo|cota de fundo|fi em cotas|fundo itau|fundo bradesco|fundo caixa)\b"
    padrao_participacao_inequivoca = r"\b(capital social da empresa|capital social de empresa|capital social|participacao societaria|participacoes societarias|participacao no capital social|participacao em empresa|participacao na empresa|capital da empresa|quota de capital social|quotas de capital social|cota de capital social|cotas de capital social|aporte de capital|capital em empresa)\b"
    padrao_rural = r"\b(gado|bovino|bovina|rebanho|semovente|boi|vaca|bezerro|novilha|animais do tipo bovino)\b"
    padrao_cripto = r"\b(bitcoin|btc|ethereum|cripto|criptoativo|criptomoeda|binance|moeda eletronica|carteira digital)\b"
    padrao_previdencia = r"\b(vgbl|pgbl|previdencia|vida gerador de beneficio|brasil prev|brasilprev|fic prev|peculio|pait|previ)\b"
    padrao_derivativos = r"\b(mercado futuro|mercado futuros|opcoes|termo|derivativo|derivativos)\b"
    padrao_ouro = r"\b(ouro|ativo financeiro ouro|ourocap|brasilcap)\b"
    padrao_fundos = r"\b(fundo|fundos|fii|fidc|fip|fic|fundo de investimento|multimercado|exec premium)\b"
    padrao_renda_fixa = r"\b(cdb|rdb|renda fixa|tesouro|selic|lci|lca|debenture|debentures|cri|cra|certificado de deposito|rf|di|cash)\b"
    padrao_acoes = r"\b(acoes|acao|ordinarias|preferenciais|b3|bolsa de valores|clear|xp investimentos|rico|modal)\b"
    padrao_poupanca = r"\b(poupanca|caderneta de poupanca)\b"
    padrao_conta_corrente = r"\b(conta corrente|deposito bancario|saldo em conta|conta bancaria|banco do brasil|bradesco|itau|santander|caixa economica|nubank|inter|agencia|c c|saldo bancario|banco sicoob|cef|disponibilidade financeira|numerario)\b"

    if re.search(padrao_fundo_forte, texto) and not re.search(padrao_participacao_inequivoca, texto):
        return "fundos_investimento"
    if re.search(padrao_participacao_inequivoca, texto):
        return "participacoes_societarias"
    if re.search(padrao_rural, texto):
        return "rural_agropecuario"
    if re.search(padrao_cripto, texto):
        return "criptoativos"
    if re.search(padrao_previdencia, texto):
        return "previdencia_vgbl"
    if re.search(padrao_derivativos, texto):
        return "mercado_derivativos"
    if re.search(padrao_ouro, texto):
        return "ouro_ativo_financeiro"
    if re.search(padrao_fundos, texto):
        return "fundos_investimento"
    if re.search(padrao_renda_fixa, texto):
        return "renda_fixa"
    if re.search(padrao_acoes, texto):
        return "acoes"
    if re.search(padrao_poupanca, texto):
        return "poupanca"
    if re.search(padrao_conta_corrente, texto):
        return "deposito_conta_corrente"
    if "aplicacao" in texto or "aplicacoes" in texto or "investimento" in texto:
        return "outros_investimentos_financeiros"
    return "financeiro_generico"


def classificar_submacro_outros(row):
    texto = f"{row['tipo_norm']} {row['desc_norm']}"

    padrao_participacao = r"\b(quotas ou quinhoes de capital|quota de capital|quotas de capital|cota de capital|cotas de capital|capital social|participacao societaria|participacoes societarias|participacao em empresa|participacao na empresa|capital da empresa|empresa .* ltda|ltda|cnpj|acoes ordinarias|acoes preferenciais)\b"
    padrao_creditos = r"\b(direitos creditorios|direito creditorio|credito|creditos|saldo a receber|valor a receber|a receber|mutuo|emprestimo|acao judicial|processo judicial|deposito judicial|judicial|precatorio|indenizacao|devolucao de imposto|imposto de renda|restituicao|dividendos a receber|heranca|direitos hereditarios|inventario|espolio)\b"
    padrao_financeiro = r"\b(conta corrente|saldo em conta|banco do brasil|banco|bradesco|itau|santander|caixa economica|cef|poupanca|aplicacao|aplicacoes|investimento|vgbl|pgbl|previdencia|fundo|fundos|cdb|rdb|renda fixa|tesouro|bitcoin|cripto|moeda corrente|dinheiro em especie|moeda nacional)\b"
    padrao_rural = r"\b(fazenda|sitio|chacara|gleba|area rural|imovel rural|propriedade rural|hectare|hectares|cabecas de gado|cabeca de gado|cabecas|cabeca|gado|bovino|bovinos|bovina|bovinas|rebanho|semovente|semoventes|boi|bois|vaca|vacas|bezerro|bezerros|novilha|novilhas|pecuaria|criacao de animais|animais bovinos|animais|implementos agricolas|implemento agricola|atividade rural|exploracao de atividade rural|terra rural|terras rurais)\b|(\bha\b|\bhas\b)"
    padrao_imoveis = r"\b(imovel|imoveis|apartamento|casa|terreno|lote|predio|sala comercial|loja|galpao|construcao|benfeitoria|area urbana|predio comercial|imovel comercial|vinicula|pousada|hotel|supermercado|padaria)\b"
    padrao_veiculos = r"\b(veiculo|automovel|carro|moto|motocicleta|caminhao|camionete|camioneta|onibus|renavam|placa|trator|lancha|barco|embarcacao|aeronave|aviao)\b"
    padrao_atividade_profissional = r"\b(atividade autonoma|exercicio da atividade autonoma|equipamento profissional|consultorio|escritorio|maquinas e equipamentos|ferramentas)\b"

    if re.search(padrao_participacao, texto):
        return "participacoes_societarias"
    if re.search(padrao_creditos, texto):
        return "creditos_direitos"
    if re.search(padrao_rural, texto):
        return "rural_agropecuario"
    if re.search(padrao_imoveis, texto):
        return "imoveis"
    if re.search(padrao_veiculos, texto):
        return "veiculos"
    if re.search(padrao_financeiro, texto):
        return "ativos_financeiros"
    if re.search(padrao_atividade_profissional, texto):
        return "outros_atividade_profissional"
    return "outros"


def classificar_submacro_veiculos(row):
    tipo = str(row["tipo_norm"])
    desc = str(row["desc_norm"])
    texto = desc

    if "aeronave" in tipo:
        return "aeronave"
    if "embarcacao" in tipo:
        return "embarcacao"

    padrao_maquina_agricola = r"\b(?:trator|tratores|colheitadeira|plantadeira|pulverizador|maquina agricola|maquinas agricolas|implemento agricola|implementos agricolas|grade aradora|arado|rocadeira|carreta agricola|plataforma de corte)\b"
    padrao_aeronave = r"\b(?:aeronave|aviao|helicoptero|ultraleve|embraer|phenom|aircraft|cessna|beech|prefixo pt|prefixo pr)\b"
    padrao_embarcacao = r"\b(?:barco|lancha|embarcacao|velereiro|veleiro|jetski|jet ski|iate|bote)\b"
    padrao_caminhao_onibus = r"\b(?:caminhao|caminhoes|onibus|microonibus|micro onibus|carreta|reboque|semi reboque|cavalinho|sprinter|ducato|kombi|furgon|furgao|volvo fh|vol vo|fh 460)\b"
    padrao_moto = r"\b(?:moto|motocicleta|motociclo|honda biz|biz|cg 150|cg150|yamaha|suzuki)\b"
    padrao_veiculo_leve = r"\b(?:carro|automovel|veiculo|veiuculo|veiuclo|camionete|camioneta|caminhonete|pickup|pick up|fiat|ford|chevrolet|toyota|honda|hyundai|hyndai|hyunday|volkswagen|vw|renault|peugeot|citroen|nissan|gm|jeep|mitsubishi|mercedes|bmw|audi|kia|volvo|land rover|range rover|range rovery|dodge|ram|amarok|hb20|seat|cordoba|tucson|hillux|hilux|gol|palio|uno|corolla|s10|ranger|civic|pajero|discovery|compass|sw4|onix|prisma|sandero|logan|ecosport|fox|saveiro|strada|toro|spin|cruze|ka|opala)\b"

    if re.search(padrao_maquina_agricola, texto):
        return "maquina_agricola"
    if re.search(padrao_aeronave, texto):
        return "aeronave"
    if re.search(padrao_embarcacao, texto):
        return "embarcacao"
    if re.search(padrao_caminhao_onibus, texto):
        return "caminhao_onibus"
    if re.search(padrao_moto, texto):
        return "moto"
    if re.search(padrao_veiculo_leve, texto):
        return "veiculo_leve"

    return "veiculo_generico"

## 5. Pipeline consolidado de tratamento

A função `tratar_bens` aplica as regras em ordem controlada e preserva a coluna `submacro_categoria` como nome auxiliar usado nas etapas seguintes.


In [5]:
def aplicar_regras_manuais(df, regras):
    """Aplica regras manuais definidas no topo do notebook."""
    df = df.copy()

    for regra in regras:
        nome = regra.get("nome", "regra_sem_nome")
        padrao = regra["padrao"]
        macro = regra.get("macro")
        submacro_col = regra.get("submacro_col")
        submacro_valor = regra.get("submacro_valor")

        mask = df["texto_norm"].str.contains(padrao, regex=True, na=False)

        if macro is not None:
            df.loc[mask, "macro_categoria_refinada"] = macro

        if submacro_col is not None and submacro_valor is not None:
            if submacro_col not in df.columns:
                df[submacro_col] = np.nan
            df.loc[mask, submacro_col] = submacro_valor

        df.loc[mask, "regra_manual_aplicada"] = (
            df.loc[mask, "regra_manual_aplicada"].fillna("").astype(str)
            + (";" + nome)
        ).str.strip(";")

    return df


def tratar_bens(df, params, regras_manuais=None):
    """Executa a versão final do tratamento dos bens declarados."""
    regras_manuais = regras_manuais or []

    df = preparar_base_inicial(df, params)
    df["regra_manual_aplicada"] = np.nan

    # 1. Imóveis
    df["submacro_imoveis"] = pd.Series(pd.NA, index=df.index, dtype="object")
    mask_imoveis = df["macro_categoria"] == "imoveis"
    df.loc[mask_imoveis, "submacro_imoveis"] = (
        df.loc[mask_imoveis].apply(classificar_submacro_imoveis, axis=1)
    )
    df.loc[mask_imoveis & (df["submacro_imoveis"] == "imovel_rural"), "macro_categoria_refinada"] = "rural_agropecuario"
    df.loc[mask_imoveis & (df["submacro_imoveis"] == "participacoes_societarias"), "macro_categoria_refinada"] = "participacoes_societarias"

    # Mantém a coluna auxiliar de submacro de imóveis usada nas etapas seguintes.
    df["submacro_categoria"] = df["submacro_imoveis"]

    # 2. Ativos financeiros
    df["submacro_financeira"] = pd.Series(pd.NA, index=df.index, dtype="object")
    mask_fin = df["macro_categoria_refinada"] == "ativos_financeiros"
    df.loc[mask_fin, "submacro_financeira"] = (
        df.loc[mask_fin].apply(classificar_submacro_financeira, axis=1)
    )
    df.loc[mask_fin & (df["submacro_financeira"] == "participacoes_societarias"), "macro_categoria_refinada"] = "participacoes_societarias"
    df.loc[mask_fin & (df["submacro_financeira"] == "rural_agropecuario"), "macro_categoria_refinada"] = "rural_agropecuario"

    # 3. Outros
    df["submacro_outros"] = pd.Series(pd.NA, index=df.index, dtype="object")
    mask_outros = df["macro_categoria_refinada"] == "outros"
    df.loc[mask_outros, "submacro_outros"] = (
        df.loc[mask_outros].apply(classificar_submacro_outros, axis=1)
    )
    mask_outros_reclass = (
        (df["macro_categoria_refinada"] == "outros")
        & df["submacro_outros"].notna()
        & (df["submacro_outros"] != "outros")
    )
    df.loc[mask_outros_reclass, "macro_categoria_refinada"] = df.loc[mask_outros_reclass, "submacro_outros"]

    # 4. Veículos
    df["submacro_veiculos"] = pd.Series(pd.NA, index=df.index, dtype="object")
    mask_veiculos = df["macro_categoria_refinada"] == "veiculos"
    df.loc[mask_veiculos, "submacro_veiculos"] = (
        df.loc[mask_veiculos].apply(classificar_submacro_veiculos, axis=1)
    )

    # Correções adicionais para veículos genéricos.
    mask_veic_generico = (
        (df["macro_categoria_refinada"] == "veiculos")
        & (df["submacro_veiculos"] == "veiculo_generico")
    )
    texto_veic_generico = df["desc_norm"].astype(str)

    padrao_imovel_em_veiculos = r"\b(?:casa|alvenaria|rua|apartamento|terreno|lote|imovel|residencia|residencial|financiada pela caixa)\b"
    padrao_maquina_agricola_extra = r"\b(?:new holland|massey|ferguson|john deere|jonh deere|valtra|case ih|colheitadeira|plantadeira|maquinas e equipamentos|maquina agricola|trator|tratores|quadriciclo)\b"
    padrao_veiculo_leve_extra = r"\b(?:checrolet|chevrolet|corvette|m benz|benz|mercedes|trailblazer|traiblazer|c200|blindada|blindado|veiculos automotores|veiculo automotor|automotores|brp)\b"

    df.loc[mask_veic_generico & texto_veic_generico.str.contains(padrao_imovel_em_veiculos, regex=True, na=False), "submacro_veiculos"] = "imoveis"
    df.loc[mask_veic_generico & texto_veic_generico.str.contains(padrao_maquina_agricola_extra, regex=True, na=False), "submacro_veiculos"] = "maquina_agricola"
    df.loc[mask_veic_generico & texto_veic_generico.str.contains(padrao_veiculo_leve_extra, regex=True, na=False), "submacro_veiculos"] = "veiculo_leve"

    # Reclassificações finais vindas de veículos.
    mask_veiculos = df["macro_categoria_refinada"] == "veiculos"
    df.loc[mask_veiculos & (df["submacro_veiculos"] == "maquina_agricola"), "macro_categoria_refinada"] = "rural_agropecuario"
    df.loc[mask_veiculos & (df["submacro_veiculos"] == "imoveis"), "macro_categoria_refinada"] = "imoveis"

    # 5. Regras manuais finais
    if regras_manuais:
        df = aplicar_regras_manuais(df, regras_manuais)

    return df


## 6. Geração das features por candidato

Esta seção gera o dataset final de features patrimoniais. A saída segue sempre a estrutura definida em `MACROS_FEATURES_PADRAO`: `SQ_CANDIDATO`, valores por macro, `patrimonio_total`, percentuais por macro, métricas auxiliares e metadados como `SG_UF`.


In [6]:
def gerar_features_candidatos(bens_tratados, params):
    """
    Gera uma linha por candidato com features patrimoniais.

    A ordem das colunas segue sempre MACROS_FEATURES_PADRAO:
    SQ_CANDIDATO, valores por macro, patrimonio_total, percentuais, métricas auxiliares e metadados.
    """
    df = bens_tratados.copy()

    if "SQ_CANDIDATO" not in df.columns:
        raise KeyError("A base precisa conter SQ_CANDIDATO para gerar features por candidato.")

    macro_col = "macro_categoria_refinada"
    valor_col = "VR_BEM_CANDIDATO_NUM"
    macros_padrao = params.get("MACROS_FEATURES_PADRAO", [])

    df[macro_col] = df[macro_col].fillna(params.get("VALOR_PREENCHER_MACRO_NA", "outros"))

    valor_por_macro = (
        df.pivot_table(
            index="SQ_CANDIDATO",
            columns=macro_col,
            values=valor_col,
            aggfunc="sum",
            fill_value=0
        )
        .sort_index()
    )

    # Garante as macros esperadas mesmo que alguma não apareça em uma rodada específica.
    for macro in macros_padrao:
        if macro not in valor_por_macro.columns:
            valor_por_macro[macro] = 0

    if macros_padrao:
        valor_por_macro = valor_por_macro[macros_padrao]
    else:
        valor_por_macro = valor_por_macro.reindex(sorted(valor_por_macro.columns), axis=1)

    valor_por_macro.columns = [f"valor_{col}" for col in valor_por_macro.columns]
    colunas_valor_macro = list(valor_por_macro.columns)

    features = valor_por_macro.copy()
    features["patrimonio_total"] = features[colunas_valor_macro].sum(axis=1)

    # Percentuais vêm imediatamente após patrimonio_total para manter a estrutura final padronizada.
    colunas_percentuais = []
    for col in colunas_valor_macro:
        macro = col.replace("valor_", "", 1)
        perc_col = f"perc_{macro}"
        features[perc_col] = np.where(
            features["patrimonio_total"] > 0,
            features[col] / features["patrimonio_total"],
            0
        )
        colunas_percentuais.append(perc_col)

    features["qtd_bens"] = df.groupby("SQ_CANDIDATO").size()
    features["qtd_macros_presentes"] = (features[colunas_valor_macro] > 0).sum(axis=1)
    features["indice_concentracao_macro"] = (features[colunas_percentuais] ** 2).sum(axis=1)
    features["log_patrimonio_total"] = np.log1p(features["patrimonio_total"])
    features["log_qtd_bens"] = np.log1p(features["qtd_bens"])

    # Metadados por candidato. Por padrão, só SG_UF entra no dataset final.
    metadados_features = params.get("METADADOS_FEATURES", ["SG_UF"])
    metadados_candidato = {}
    for col in metadados_features:
        if col in df.columns:
            metadados_candidato[col] = df.groupby("SQ_CANDIDATO")[col].first()

    if metadados_candidato:
        metadados = pd.concat(metadados_candidato.values(), axis=1)
        metadados.columns = list(metadados_candidato.keys())
        features = features.join(metadados)

    features = features.reset_index()

    # Ordem final explícita para reprodutibilidade.
    ordem = ["SQ_CANDIDATO"] + colunas_valor_macro + ["patrimonio_total"] + colunas_percentuais + [
        "qtd_bens",
        "qtd_macros_presentes",
        "indice_concentracao_macro",
        "log_patrimonio_total",
        "log_qtd_bens",
    ]
    ordem += [col for col in metadados_features if col in features.columns]
    features = features[[col for col in ordem if col in features.columns]].copy()

    if params["REMOVER_CANDIDATOS_PATRIMONIO_ZERO_FEATURES"]:
        features = features[features["patrimonio_total"] > 0].copy()

    return features


def preparar_bens_modelagem(bens_tratados):
    """
    Seleciona colunas úteis para a próxima fase do pipeline.
    Mantém a coluna auxiliar submacro_categoria.
    """
    colunas_preferenciais = [
        "DT_GERACAO",
        "SG_UF",
        "SQ_CANDIDATO",
        "CD_TIPO_BEM_CANDIDATO",
        "DS_TIPO_BEM_CANDIDATO",
        "DS_BEM_CANDIDATO",
        "VR_BEM_CANDIDATO",
        "macro_categoria",
        "perc_registros",
        "VR_BEM_CANDIDATO_NUM",
        "desc_norm",
        "tipo_norm",
        "submacro_categoria",
        "macro_categoria_refinada",
        "submacro_financeira",
        "submacro_outros",
        "submacro_veiculos",
        "arquivo_origem",
    ]

    colunas = [col for col in colunas_preferenciais if col in bens_tratados.columns]
    return bens_tratados[colunas].copy()


## 7. Auditoria da rodada

A auditoria permite revisar se o tratamento está satisfatório antes de passar para a fase de clusterização.

In [7]:
def gerar_auditoria(bens_tratados, features_candidatos, params, origem):
    """Gera tabelas de auditoria da rodada."""
    n = params["N_AMOSTRAS_AUDITORIA"]
    valor_min = params["VALOR_MINIMO_AUDITORIA"]

    auditoria = {}

    auditoria["origem"] = pd.DataFrame([{
        "origem": origem,
        "linhas_bens_tratados": len(bens_tratados),
        "candidatos_features": len(features_candidatos),
        "rodada": params["NOME_RODADA"],
        "gerado_em": datetime.now().isoformat(timespec="seconds")
    }])

    tipos_base = pd.Series(MAPA_TIPO_MACRO).rename("macro_categoria").reset_index()
    tipos_base.columns = ["DS_TIPO_BEM_CANDIDATO", "macro_categoria_mapeada"]
    auditoria["mapa_tipos_macro"] = tipos_base

    tipos_sem_macro = (
        bens_tratados.loc[
            ~bens_tratados["DS_TIPO_BEM_CANDIDATO"].isin(MAPA_TIPO_MACRO.keys()),
            "DS_TIPO_BEM_CANDIDATO"
        ]
        .drop_duplicates()
        .sort_values()
        .to_frame("DS_TIPO_BEM_CANDIDATO")
    )
    auditoria["tipos_sem_macro_explicita"] = tipos_sem_macro

    auditoria["resumo_macro_refinada"] = resumo_categoria(bens_tratados)

    for macro, subcol in [
        ("imoveis", "submacro_imoveis"),
        ("ativos_financeiros", "submacro_financeira"),
        ("outros", "submacro_outros"),
        ("veiculos", "submacro_veiculos"),
    ]:
        if subcol in bens_tratados.columns:
            mask = bens_tratados["macro_categoria"] == macro
            if macro in ["outros", "veiculos"]:
                mask = bens_tratados["macro_categoria_refinada"] == macro

            auditoria[f"resumo_{subcol}"] = resumo_categoria(
                bens_tratados.loc[mask].rename(columns={subcol: "categoria_auditoria"}),
                categoria_col="categoria_auditoria"
            )

    # Amostras para revisão manual.
    colunas_amostra = [
        col for col in [
            "SQ_CANDIDATO",
            "SG_UF",
            "DS_TIPO_BEM_CANDIDATO",
            "DS_BEM_CANDIDATO",
            "VR_BEM_CANDIDATO_NUM",
            "macro_categoria",
            "macro_categoria_refinada",
            "submacro_imoveis",
            "submacro_financeira",
            "submacro_outros",
            "submacro_veiculos",
            "regra_manual_aplicada",
        ]
        if col in bens_tratados.columns
    ]

    masks_amostra = {
        "amostra_outros_residual": bens_tratados["macro_categoria_refinada"].eq("outros"),
        "amostra_financeiro_generico": bens_tratados.get("submacro_financeira", pd.Series(False, index=bens_tratados.index)).eq("financeiro_generico"),
        "amostra_imovel_generico": bens_tratados.get("submacro_imoveis", pd.Series(False, index=bens_tratados.index)).eq("imovel_generico"),
        "amostra_veiculo_generico": bens_tratados.get("submacro_veiculos", pd.Series(False, index=bens_tratados.index)).eq("veiculo_generico"),
    }

    for nome, mask in masks_amostra.items():
        auditoria[nome] = (
            bens_tratados.loc[mask & (bens_tratados["VR_BEM_CANDIDATO_NUM"] >= valor_min), colunas_amostra]
            .sort_values("VR_BEM_CANDIDATO_NUM", ascending=False)
            .head(n)
        )

    auditoria["features_describe"] = features_candidatos.describe(include="all").T.reset_index().rename(columns={"index": "coluna"})

    percentuais = [col for col in features_candidatos.columns if col.startswith("perc_")]
    if percentuais:
        soma_percentuais = features_candidatos[percentuais].sum(axis=1)
        auditoria["auditoria_soma_percentuais"] = soma_percentuais.describe().to_frame("soma_percentuais")

    auditoria["valores_faltantes_features"] = (
        features_candidatos.isna().sum().sort_values(ascending=False).to_frame("qtd_nulos")
    )

    if "SQ_CANDIDATO" in features_candidatos.columns:
        auditoria["duplicados_features"] = pd.DataFrame([{
            "candidatos_duplicados": int(features_candidatos["SQ_CANDIDATO"].duplicated().sum())
        }])

    return auditoria


def avaliar_checklist(auditoria, checklist):
    """Compara indicadores principais com limites sugeridos."""
    resumo = auditoria.get("resumo_macro_refinada", pd.DataFrame())
    linhas = []

    def valor_macro(macro, col):
        if resumo.empty or macro not in resumo.index:
            return 0
        return float(resumo.loc[macro, col])

    linhas.append({
        "criterio": "perc_qtd_outros",
        "valor": valor_macro("outros", "perc_qtd"),
        "limite": checklist["max_perc_macro_outros"],
        "ok": valor_macro("outros", "perc_qtd") <= checklist["max_perc_macro_outros"],
    })

    linhas.append({
        "criterio": "perc_valor_outros",
        "valor": valor_macro("outros", "perc_valor_total"),
        "limite": checklist["max_perc_valor_outros"],
        "ok": valor_macro("outros", "perc_valor_total") <= checklist["max_perc_valor_outros"],
    })

    for nome_tabela, criterio, limite_key in [
        ("resumo_submacro_financeira", "perc_qtd_financeiro_generico", "max_perc_financeiro_generico"),
        ("resumo_submacro_imoveis", "perc_qtd_imovel_generico", "max_perc_imovel_generico"),
        ("resumo_submacro_veiculos", "perc_qtd_veiculo_generico", "max_perc_veiculo_generico"),
    ]:
        tab = auditoria.get(nome_tabela, pd.DataFrame())
        categoria = criterio.replace("perc_qtd_", "")
        valor = 0
        if not tab.empty and categoria in tab.index:
            valor = float(tab.loc[categoria, "perc_qtd"])
        linhas.append({
            "criterio": criterio,
            "valor": valor,
            "limite": checklist[limite_key],
            "ok": valor <= checklist[limite_key],
        })

    tipos_sem_macro = auditoria.get("tipos_sem_macro_explicita", pd.DataFrame())
    qtd_sem_macro = len(tipos_sem_macro)
    linhas.append({
        "criterio": "qtd_tipos_sem_macro_explicita",
        "valor": qtd_sem_macro,
        "limite": checklist["max_tipos_sem_macro"],
        "ok": qtd_sem_macro <= checklist["max_tipos_sem_macro"],
    })

    return pd.DataFrame(linhas)

## 8. Exportação dos resultados

Cada rodada gera uma pasta própria. A seleção dos arquivos finais é controlada por `PARAMS["ARQUIVOS_SAIDA"]`.

Não há comparação com arquivo de referência nesta versão.


In [8]:
def saida_ativa(params, nome):
    """Indica se um arquivo ou grupo de saída deve ser gerado."""
    return params.get("ARQUIVOS_SAIDA", {}).get(nome, True)


def exportar_resultados(bens_modelagem, features_candidatos, auditoria, checklist_df, params):
    """Salva os resultados selecionados da rodada em pasta versionada."""
    dir_saida = Path(params["DIR_SAIDA_BASE"]) / params["NOME_RODADA"]
    dir_saida.mkdir(parents=True, exist_ok=True)

    sep = params["SEP_SAIDA"]
    enc = params["ENCODING_SAIDA"]

    caminhos = {}

    if saida_ativa(params, "bens_tratados"):
        caminhos["bens_tratados"] = dir_saida / "bens_candidatos_tratados.csv"
        bens_modelagem.to_csv(caminhos["bens_tratados"], sep=sep, index=False, encoding=enc)

    if saida_ativa(params, "bens_limpo_com_macros"):
        caminhos["bens_limpo_com_macros"] = dir_saida / "bens_candidatos_limpo_com_macros.csv"
        colunas_limpo = [
            "SG_UF",
            "SQ_CANDIDATO",
            "DS_TIPO_BEM_CANDIDATO",
            "DS_BEM_CANDIDATO",
            "VR_BEM_CANDIDATO",
            "macro_categoria_refinada",
        ]
        colunas_limpo = [col for col in colunas_limpo if col in bens_modelagem.columns]
        bens_modelagem[colunas_limpo].to_csv(caminhos["bens_limpo_com_macros"], sep=sep, index=False, encoding=enc)

    if saida_ativa(params, "features_candidatos"):
        caminhos["features_candidatos"] = dir_saida / "features_patrimoniais_por_candidato.csv"
        features_candidatos.to_csv(caminhos["features_candidatos"], sep=sep, index=False, encoding=enc)

    if saida_ativa(params, "features_final_raiz"):
        caminhos["features_final_raiz"] = Path("features_patrimoniais_por_candidato.csv")
        features_candidatos.to_csv(caminhos["features_final_raiz"], sep=sep, index=False, encoding=enc)

    if saida_ativa(params, "checklist"):
        caminhos["checklist"] = dir_saida / "checklist_rodada.csv"
        checklist_df.to_csv(caminhos["checklist"], sep=sep, index=False, encoding=enc)

    if saida_ativa(params, "auditorias"):
        dir_auditoria = dir_saida / "auditoria"
        dir_auditoria.mkdir(exist_ok=True)

        for nome, tabela in auditoria.items():
            if isinstance(tabela, pd.DataFrame):
                caminho = dir_auditoria / f"{nome}.csv"
                tabela.to_csv(caminho, sep=sep, index=True, encoding=enc)
                caminhos[f"auditoria_{nome}"] = caminho

    if saida_ativa(params, "config"):
        config_export = {
            "PARAMS": params,
            "AJUSTES_TIPO_MACRO": AJUSTES_TIPO_MACRO,
            "REGRAS_MANUAIS_TEXTO": REGRAS_MANUAIS_TEXTO,
            "CHECKLIST_ACEITE": CHECKLIST_ACEITE,
        }

        caminhos["config"] = dir_saida / "config_rodada.json"
        with open(caminhos["config"], "w", encoding="utf-8") as f:
            json.dump(config_export, f, ensure_ascii=False, indent=2)

    if saida_ativa(params, "zip_rodada"):
        zip_path = Path(params["DIR_SAIDA_BASE"]) / f"{params['NOME_RODADA']}.zip"
        if zip_path.exists():
            zip_path.unlink()
        shutil.make_archive(str(zip_path).replace(".zip", ""), "zip", dir_saida)
        caminhos["zip_rodada"] = zip_path

    return caminhos


def executar_pipeline(params, regras_manuais=None):
    """Executa a rodada completa do pipeline."""
    df_raw, origem = ler_base_bens(params)
    bens_tratados = tratar_bens(df_raw, params, regras_manuais=regras_manuais)
    bens_modelagem = preparar_bens_modelagem(bens_tratados)
    features_candidatos = gerar_features_candidatos(bens_tratados, params)
    auditoria = gerar_auditoria(bens_tratados, features_candidatos, params, origem)
    checklist_df = avaliar_checklist(auditoria, CHECKLIST_ACEITE)
    caminhos = exportar_resultados(bens_modelagem, features_candidatos, auditoria, checklist_df, params)

    return {
        "origem": origem,
        "bens_tratados": bens_tratados,
        "bens_modelagem": bens_modelagem,
        "features_candidatos": features_candidatos,
        "auditoria": auditoria,
        "checklist": checklist_df,
        "caminhos": caminhos,
    }


## 9. Execução e exportação

Execute esta célula para rodar o tratamento completo. Para uma nova rodada, altere `NOME_RODADA`, `CAMINHO_BENS_UNIFICADO`, as regras no topo e/ou a seleção de `ARQUIVOS_SAIDA`.


In [9]:
resultado = executar_pipeline(
    PARAMS,
    regras_manuais=REGRAS_MANUAIS_TEXTO
)

print("Origem:", resultado["origem"])
print("Bens tratados:", resultado["bens_tratados"].shape)
print("Bens para modelagem:", resultado["bens_modelagem"].shape)
print("Features por candidato:", resultado["features_candidatos"].shape)
print("\nArquivos gerados:")
for nome, caminho in resultado["caminhos"].items():
    print(f"- {nome}: {caminho}")


Origem: arquivo_unificado:bem_candidato_2022_BRASIL.csv | encoding=latin1 | sep=;
Bens tratados: (92538, 32)
Bens para modelagem: (92538, 17)
Features por candidato: (18219, 30)

Arquivos gerados:
- bens_tratados: features_patrimoniais_por_candidato/bens_candidatos_tratados.csv
- bens_limpo_com_macros: features_patrimoniais_por_candidato/bens_candidatos_limpo_com_macros.csv
- features_candidatos: features_patrimoniais_por_candidato/features_patrimoniais_por_candidato.csv
- features_final_raiz: features_patrimoniais_por_candidato.csv
- checklist: features_patrimoniais_por_candidato/checklist_rodada.csv
- auditoria_origem: features_patrimoniais_por_candidato/auditoria/origem.csv
- auditoria_mapa_tipos_macro: features_patrimoniais_por_candidato/auditoria/mapa_tipos_macro.csv
- auditoria_tipos_sem_macro_explicita: features_patrimoniais_por_candidato/auditoria/tipos_sem_macro_explicita.csv
- auditoria_resumo_macro_refinada: features_patrimoniais_por_candidato/auditoria/resumo_macro_refinada

In [10]:
# Resumo dos arquivos efetivamente gerados nesta rodada.
pd.DataFrame(
    [{"saida": nome, "caminho": str(caminho)} for nome, caminho in resultado["caminhos"].items()]
)


,saida,caminho
0,bens_tratados,features_patrimoniais_por_candidato/bens_candi...
1,bens_limpo_com_macros,features_patrimoniais_por_candidato/bens_candi...
2,features_candidatos,features_patrimoniais_por_candidato/features_p...
3,features_final_raiz,features_patrimoniais_por_candidato.csv
4,checklist,features_patrimoniais_por_candidato/checklist_...
5,auditoria_origem,features_patrimoniais_por_candidato/auditoria/...
6,auditoria_mapa_tipos_macro,features_patrimoniais_por_candidato/auditoria/...
7,auditoria_tipos_sem_macro_explicita,features_patrimoniais_por_candidato/auditoria/...
8,auditoria_resumo_macro_refinada,features_patrimoniais_por_candidato/auditoria/...
9,auditoria_resumo_submacro_imoveis,features_patrimoniais_por_candidato/auditoria/...


## 10. Revisão rápida da rodada

Use as tabelas abaixo para decidir se o tratamento está satisfatório. Se algum ponto não estiver bom, volte aos parâmetros/regras no topo, altere `NOME_RODADA` e reexecute.


In [11]:
resultado["checklist"]

,criterio,valor,limite,ok
0,perc_qtd_outros,2.83,10.0,True
1,perc_valor_outros,1.52,10.0,True
2,perc_qtd_financeiro_generico,0.00,20.0,True
3,perc_qtd_imovel_generico,2.87,20.0,True
4,perc_qtd_veiculo_generico,9.66,20.0,True
5,qtd_tipos_sem_macro_explicita,0.00,0.0,True


In [12]:
resultado["auditoria"]["resumo_macro_refinada"]

,qtd,valor_total,valor_mediano,valor_medio,perc_valor_total,perc_qtd
macro_categoria_refinada,,,,,,
participacoes_societarias,8960,281532058545,500000.0,3.142099e+07,34.51,9.68
ativos_financeiros,24873,195484572873,261922.0,7.859308e+06,23.96,26.88
imoveis,27973,195208369366,1770800.0,6.978457e+06,23.93,30.23
creditos_direitos,2449,55777892212,1400000.0,2.277578e+07,6.84,2.65
rural_agropecuario,5218,46879139140,1319273.0,8.984120e+06,5.75,5.64
veiculos,17612,19564701105,430000.0,1.110873e+06,2.40,19.03
outros,2623,12425687978,300000.0,4.737205e+06,1.52,2.83
dinheiro_especie,2485,7382134535,450000.0,2.970678e+06,0.90,2.69
bens_luxo_colecao,224,1330031489,205240.0,5.937641e+06,0.16,0.24


In [13]:
resultado["features_candidatos"].head()

,SQ_CANDIDATO,valor_ativos_financeiros,valor_bens_luxo_colecao,valor_creditos_direitos,valor_dinheiro_especie,valor_direitos_intangiveis,valor_imoveis,valor_outros,valor_outros_atividade_profissional,valor_participacoes_societarias,...,perc_outros_atividade_profissional,perc_participacoes_societarias,perc_rural_agropecuario,perc_veiculos,qtd_bens,qtd_macros_presentes,indice_concentracao_macro,log_patrimonio_total,log_qtd_bens,SG_UF
0,10001595335,0,0,0,0,0,0,0,0,0,...,0.0,0.0,0.0,1.000000,2,1,1.000000,13.304687,1.098612,AC
1,10001595336,0,0,0,0,0,4000000,0,0,0,...,0.0,0.0,0.0,0.221638,3,2,0.654970,15.452369,1.386294,AC
2,10001595338,0,0,0,0,0,3000000,0,0,0,...,0.0,0.0,0.0,0.038462,2,2,0.926036,14.953344,1.098612,AC
3,10001595339,0,0,0,0,0,2500000,0,0,0,...,0.0,0.0,0.0,0.285714,2,2,0.591837,15.068274,1.098612,AC
4,10001595340,0,0,0,0,0,0,0,0,0,...,0.0,0.0,0.0,1.000000,2,1,1.000000,12.676079,1.098612,AC


In [14]:
resultado["auditoria"]["origem"]


,origem,linhas_bens_tratados,candidatos_features,rodada,gerado_em
0,arquivo_unificado:bem_candidato_2022_BRASIL.cs...,92538,18219,features_patrimoniais_por_candidato,2026-05-13T13:38:29


## 11. Auditorias para ajuste iterativo

As amostras abaixo ajudam a decidir se novas regras precisam ser adicionadas em `REGRAS_MANUAIS_TEXTO` ou `AJUSTES_TIPO_MACRO`.

In [15]:
resultado["auditoria"].get("amostra_outros_residual", pd.DataFrame()).head(PARAMS["N_AMOSTRAS_AUDITORIA"])

,SQ_CANDIDATO,SG_UF,DS_TIPO_BEM_CANDIDATO,DS_BEM_CANDIDATO,VR_BEM_CANDIDATO_NUM,macro_categoria,macro_categoria_refinada,submacro_imoveis,submacro_financeira,submacro_outros,submacro_veiculos,regra_manual_aplicada
91637,250001676832,SP,OUTROS BENS E DIREITOS,Patrimônio descrito na Declaração Anual de IRP...,905103403,outros,outros,<NA>,<NA>,outros,<NA>,NaN
79205,130001619772,MG,OUTROS BENS E DIREITOS,BRASIL,890704282,outros,outros,<NA>,<NA>,outros,<NA>,NaN
66244,250001604829,SP,OUTROS BENS E DIREITOS,PROCESSO NR 0030927-64.2014.8.14.0301 - VALOR ...,424119385,outros,outros,<NA>,<NA>,outros,<NA>,NaN
66248,250001604829,SP,OUTROS BENS E DIREITOS,PROCESSO NR 0066145-56.2014.8.14.0301 - VALOR ...,360211379,outros,outros,<NA>,<NA>,outros,<NA>,NaN
87601,220001713586,RO,OUTROS BENS E DIREITOS,OUTROS PATRIMÔNIOS ESTÃO\nDECLARADOS EM NOME D...,337494412,outros,outros,<NA>,<NA>,outros,<NA>,NaN
92067,210001620215,RS,OUTROS BENS E DIREITOS,DRUMM MALHAS,289245301,outros,outros,<NA>,<NA>,outros,<NA>,NaN
8428,60001635818,CE,OUTROS BENS E DIREITOS,debentures,277914413,outros,outros,<NA>,<NA>,outros,<NA>,NaN
27773,190001644962,RJ,OUTROS BENS E DIREITOS,Empresa Corumandel,220027345,outros,outros,<NA>,<NA>,outros,<NA>,NaN
40542,250001611896,SP,OUTROS BENS E DIREITOS,AFAC NA PROVENCE PROPRIETE IMMOBILIERE INC,194301276,outros,outros,<NA>,<NA>,outros,<NA>,NaN
90461,250001611331,SP,OUTROS BENS E DIREITOS,EXECUTIVE FIC,189868732,outros,outros,<NA>,<NA>,outros,<NA>,NaN


In [16]:
resultado["auditoria"].get("amostra_financeiro_generico", pd.DataFrame()).head(PARAMS["N_AMOSTRAS_AUDITORIA"])

,SQ_CANDIDATO,SG_UF,DS_TIPO_BEM_CANDIDATO,DS_BEM_CANDIDATO,VR_BEM_CANDIDATO_NUM,macro_categoria,macro_categoria_refinada,submacro_imoveis,submacro_financeira,submacro_outros,submacro_veiculos,regra_manual_aplicada


In [17]:
resultado["auditoria"].get("amostra_imovel_generico", pd.DataFrame()).head(PARAMS["N_AMOSTRAS_AUDITORIA"])

,SQ_CANDIDATO,SG_UF,DS_TIPO_BEM_CANDIDATO,DS_BEM_CANDIDATO,VR_BEM_CANDIDATO_NUM,macro_categoria,macro_categoria_refinada,submacro_imoveis,submacro_financeira,submacro_outros,submacro_veiculos,regra_manual_aplicada
41435,250001635941,SP,Outros bens imóveis,BVI,571349805,imoveis,imoveis,imovel_generico,<NA>,<NA>,<NA>,NaN
67570,250001611446,SP,Outros bens imóveis,IMOVEL,276756848,imoveis,imoveis,imovel_generico,<NA>,<NA>,<NA>,NaN
70156,280001607834,BR,Outros bens imóveis,1/3 DE: IMÓVEIS DE HERANÇA - SANTO ANDRÉ-SP,203178056,imoveis,imoveis,imovel_generico,<NA>,<NA>,<NA>,NaN
22270,140001621343,PA,Outros bens imóveis,IMOVEL EDIFICADO AV MAGALHÃES BARATA 554 BELÉM-PA,152676576,imoveis,imoveis,imovel_generico,<NA>,<NA>,<NA>,NaN
61844,190001636390,RJ,Outros bens imóveis,OBTIDO DE INVENTÁRIO,145016928,imoveis,imoveis,imovel_generico,<NA>,<NA>,<NA>,NaN
37510,250001609961,SP,Outros bens imóveis,um imovel financiado,133929191,imoveis,imoveis,imovel_generico,<NA>,<NA>,<NA>,NaN
40561,250001597686,SP,Outros bens imóveis,Bens Imóveis declarados junto a Receita Federa...,132274292,imoveis,imoveis,imovel_generico,<NA>,<NA>,<NA>,NaN
38149,250001610005,SP,Outros bens imóveis,IMÓVEL - RIBEIRÃO PRETO - SP,114781296,imoveis,imoveis,imovel_generico,<NA>,<NA>,<NA>,NaN
5234,60001614155,CE,Outros bens imóveis,IMÓVEL FINANCIADO PELO BRADESCO,102767333,imoveis,imoveis,imovel_generico,<NA>,<NA>,<NA>,NaN
84404,190001613505,RJ,Outros bens imóveis,"1/3 DO IMOVEL SITO A ESCADA FLORA MAY 177,\nJO...",78542333,imoveis,imoveis,imovel_generico,<NA>,<NA>,<NA>,NaN


In [18]:
resultado["auditoria"].get("amostra_veiculo_generico", pd.DataFrame()).head(PARAMS["N_AMOSTRAS_AUDITORIA"])

,SQ_CANDIDATO,SG_UF,DS_TIPO_BEM_CANDIDATO,DS_BEM_CANDIDATO,VR_BEM_CANDIDATO_NUM,macro_categoria,macro_categoria_refinada,submacro_imoveis,submacro_financeira,submacro_outros,submacro_veiculos,regra_manual_aplicada
15008,100001639639,MA,"Veículo automotor terrestre: caminhão, automóv...",ESCAVADEIRA HIDRAULICA SOTREQ NF 47417,60766326,veiculos,veiculos,<NA>,<NA>,<NA>,veiculo_generico,NaN
15059,130001603578,MG,"Veículo automotor terrestre: caminhão, automóv...",S-10 2021,20642256,veiculos,veiculos,<NA>,<NA>,<NA>,veiculo_generico,NaN
31808,190001654231,RJ,"Veículo automotor terrestre: caminhão, automóv...","CONSORCIO CONTEMPLADO VALOR PAGO 97.388,33",14523602,veiculos,veiculos,<NA>,<NA>,<NA>,veiculo_generico,NaN
79512,130001634078,MG,"Veículo automotor terrestre: caminhão, automóv...",MITSUBISH OUTILANDER,14256215,veiculos,veiculos,<NA>,<NA>,<NA>,veiculo_generico,NaN
33698,210001602905,RS,"Veículo automotor terrestre: caminhão, automóv...","RAVA 2.0 4x2 TOP, ANO/MOD 2014/2015, adquirida...",14130864,veiculos,veiculos,<NA>,<NA>,<NA>,veiculo_generico,NaN
27356,190001644796,RJ,"Veículo automotor terrestre: caminhão, automóv...",MMC/OUTLANDER 2020,13583282,veiculos,veiculos,<NA>,<NA>,<NA>,veiculo_generico,NaN
54781,120001652595,MS,"Veículo automotor terrestre: caminhão, automóv...","CHEV/TRACKER ANO 2021 MODELO 2022, ALIENAÇÃO F...",13454048,veiculos,veiculos,<NA>,<NA>,<NA>,veiculo_generico,NaN
84954,190001619411,RJ,"Veículo automotor terrestre: caminhão, automóv...",Eclipse Cross HPE,12603968,veiculos,veiculos,<NA>,<NA>,<NA>,veiculo_generico,NaN
74357,80001651641,ES,"Veículo automotor terrestre: caminhão, automóv...",Renagade LNGTD/2020,12433519,veiculos,veiculos,<NA>,<NA>,<NA>,veiculo_generico,NaN
87250,220001635965,RO,"Veículo automotor terrestre: caminhão, automóv...",AQUIS. CAPTIVA SPORT - PRATA- 2010/2011- CHASS...,11921216,veiculos,veiculos,<NA>,<NA>,<NA>,veiculo_generico,NaN
